In [ ]:
## import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from tabulate import tabulate
import time
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

class DisasterPredictor:
    def __init__(self):
        self.model = None
        self.scaler = None
        self.label_encoder = None
        self.feature_names = None
        self.best_model_name = None
        self.models = {}
        self.X_test_scaled = None
        self.y_test = None
        
    def load_data(self, file_path):
        """Load dataset from an Excel file with enhanced error handling."""
        try:
            print(f"⏳ Loading data from {file_path}...")
            start_time = time.time()
            data = pd.read_excel(file_path)
            load_time = time.time() - start_time
            print(f"✅ Data loaded successfully! ({load_time:.2f} seconds)")
            print(f"📊 Dataset shape: {data.shape}")
            return data
        except Exception as e:
            print(f"❌ Error loading file: {e}")
            return None

    def preprocess_data(self, data):
        """Preprocess dataset with detailed reporting."""
        print("\n🔧 Preprocessing data...")
        
        # Handle missing values
        missing_values = data.isnull().sum()
        if missing_values.any():
            print("\n⚠️ Missing values found:")
            print(missing_values[missing_values > 0])
            print("\n🔧 Filling missing values with medians...")
            data.fillna(data.median(numeric_only=True), inplace=True)
        
        # Select features
        features = ['Temperature (°C)', 'Humidity (%)', 'Wind Speed (km/h)', 
                   'Precipitation (mm)', 'Magnitude', 'Depth (km)', 
                   'River Level (m)', 'Soil Moisture (%)', 'Atmospheric Pressure (hPa)']
        
        X = data[features]
        y = data['Disaster Type']
        
        print("\n📋 Selected Features:")
        print(tabulate([[f] for f in features], headers=['Feature'], tablefmt='pretty'))
        
        return X, y, features

    def encode_target(self, y):
        """Encode the target variable with detailed class distribution."""
        print("\n🎯 Encoding target variable...")
        label_encoder = LabelEncoder()
        y_encoded = label_encoder.fit_transform(y)
        
        # Display class distribution
        class_dist = pd.Series(y_encoded).value_counts()
        print("\n📊 Class Distribution (Before Balancing):")
        for idx, count in class_dist.items():
            print(f"{label_encoder.classes_[idx]}: {count} samples ({count/len(y_encoded):.1%})")
        
        return y_encoded, label_encoder

    def split_data(self, X, y):
        """Split data with stratification and reporting."""
        print("\n✂️ Splitting data into train/test sets...")
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y)
        
        print(f"\n📊 Training set size: {X_train.shape[0]} samples")
        print(f"📊 Test set size: {X_test.shape[0]} samples")
        
        return X_train, X_test, y_train, y_test

    def normalize_features(self, X_train, X_test):
        """Normalize features with progress reporting."""
        print("\n📏 Normalizing features...")
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        print("✅ Feature normalization complete!")
        return X_train_scaled, X_test_scaled, scaler

    def balance_data(self, X, y):
        """Apply SMOTE with detailed before/after comparison."""
        print("\n⚖️ Balancing dataset using SMOTE...")
        
        # Before balancing
        unique_before, counts_before = np.unique(y, return_counts=True)
        
        # Apply SMOTE
        smote = SMOTE(random_state=42)
        X_resampled, y_resampled = smote.fit_resample(X, y)
        
        # After balancing
        unique_after, counts_after = np.unique(y_resampled, return_counts=True)
        
        # Display comparison
        print("\n📊 Class Distribution Comparison:")
        comparison = []
        for i, (class_before, count_before) in enumerate(zip(unique_before, counts_before)):
            comparison.append([
                i,
                count_before,
                counts_after[i],
                f"+{counts_after[i] - count_before}",
                f"{count_before/len(y):.1%} → {counts_after[i]/len(y_resampled):.1%}"
            ])
        
        print(tabulate(comparison, 
                     headers=["Class", "Before", "After", "Change", "Percentage"], 
                     tablefmt="pretty"))
        
        return X_resampled, y_resampled

    def train_model(self, X_train, y_train):
        """Train Random Forest with detailed progress and results."""
        print("\n🌲 Training Random Forest Classifier...")
        
        param_dist = {
            'n_estimators': [100, 300, 500, 700],
            'max_depth': [10, 20, 30, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'class_weight': ['balanced']
        }
        
        rf = RandomForestClassifier(random_state=42)
        random_search = RandomizedSearchCV(
            rf, param_distributions=param_dist, 
            n_iter=10, cv=3, n_jobs=-1, verbose=1)
        
        start_time = time.time()
        random_search.fit(X_train, y_train)
        train_time = time.time() - start_time
        
        print(f"\n✅ Training complete! ({train_time:.2f} seconds)")
        print("\n🏆 Best Parameters:")
        for param, value in random_search.best_params_.items():
            print(f"{param}: {value}")
        
        self.models['Random Forest'] = random_search.best_estimator_
        return random_search.best_estimator_

    def train_xgboost(self, X_train, y_train):
        """Train XGBoost with detailed progress."""
        print("\n🚀 Training XGBoost Classifier...")
        
        xgb_model = XGBClassifier(
            n_estimators=300, 
            max_depth=15, 
            learning_rate=0.1, 
            random_state=42,
            eval_metric='mlogloss',
            use_label_encoder=False)
        
        start_time = time.time()
        xgb_model.fit(X_train, y_train)
        train_time = time.time() - start_time
        
        print(f"\n✅ Training complete! ({train_time:.2f} seconds)")
        self.models['XGBoost'] = xgb_model
        return xgb_model

    def evaluate_model(self, model, X_test, y_test, model_name=""):
        """Enhanced model evaluation with visualizations."""
        print(f"\n🧪 Evaluating {model_name} model...")
        
        # Make predictions
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        # Classification report
        print("\n📝 Classification Report:")
        print(classification_report(y_test, y_pred, target_names=self.label_encoder.classes_))
        
        # Confusion matrix
        print("\n📊 Confusion Matrix:")
        cm = confusion_matrix(y_test, y_pred)
        self.plot_confusion_matrix(cm, model_name)
        
        # Accuracy
        accuracy = accuracy_score(y_test, y_pred)
        print(f"\n🎯 Accuracy: {accuracy:.2%}")
        
        # Feature importance (if available)
        if hasattr(model, 'feature_importances_'):
            print("\n🔍 Feature Importance:")
            self.plot_feature_importance(model)
        
        return accuracy

    def plot_confusion_matrix(self, cm, model_name=""):
        """Plot a beautiful confusion matrix."""
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                   xticklabels=self.label_encoder.classes_,
                   yticklabels=self.label_encoder.classes_)
        plt.title(f'Confusion Matrix - {model_name}', pad=20)
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        plt.xticks(rotation=45)
        plt.yticks(rotation=0)
        plt.tight_layout()
        plt.show()

    def plot_feature_importance(self, model):
        """Plot feature importance with nice formatting."""
        importance = model.feature_importances_
        indices = np.argsort(importance)[::-1]
        
        plt.figure(figsize=(12, 6))
        plt.title("Feature Importance", pad=20)
        bars = plt.bar(range(len(self.feature_names)), importance[indices], align="center")
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}',
                   ha='center', va='bottom')
        
        plt.xticks(range(len(self.feature_names)), 
                 [self.feature_names[i] for i in indices], rotation=45)
        plt.xlim([-1, len(self.feature_names)])
        plt.tight_layout()
        plt.show()

    def compare_models(self):
        """Compare performance of all trained models."""
        if not self.models:
            print("⚠️ No models trained yet!")
            return
        
        print("\n🏆 Model Comparison:")
        results = []
        for name, model in self.models.items():
            acc = self.evaluate_model(model, self.X_test_scaled, self.y_test, name)
            results.append([name, f"{acc:.2%}"])
        
        print("\n📈 Performance Summary:")
        print(tabulate(results, headers=["Model", "Accuracy"], tablefmt="pretty"))
        
        # Select best model
        best_model_name = max(self.models.keys(), 
                            key=lambda x: accuracy_score(
                                self.y_test, 
                                self.models[x].predict(self.X_test_scaled)))
        
        self.best_model_name = best_model_name
        self.model = self.models[best_model_name]
        print(f"\n🏅 Best Model Selected: {best_model_name}")

    def save_tools(self):
        """Save trained model and preprocessing tools with verification."""
        if not all([self.model, self.scaler, self.label_encoder, self.feature_names]):
            print("⚠️ Cannot save - not all components are trained!")
            return
        
        print("\n💾 Saving model and tools...")
        try:
            joblib.dump(self.model, "disaster_model.pkl")
            joblib.dump(self.scaler, "scaler.pkl")
            joblib.dump(self.label_encoder, "label_encoder.pkl")
            joblib.dump(self.feature_names, "feature_names.pkl")
            print("✅ Successfully saved all components!")
        except Exception as e:
            print(f"❌ Error saving files: {e}")

    def load_tools(self):
        """Load saved components with error handling."""
        print("\n📂 Loading saved components...")
        try:
            self.model = joblib.load("disaster_model.pkl")
            self.scaler = joblib.load("scaler.pkl")
            self.label_encoder = joblib.load("label_encoder.pkl")
            self.feature_names = joblib.load("feature_names.pkl")
            print("✅ Successfully loaded all components!")
            return True
        except Exception as e:
            print(f"❌ Error loading files: {e}")
            return False

    def predict_disaster(self):
        """Interactive prediction with original probabilities"""
        if not all([self.model, self.scaler, self.label_encoder, self.feature_names]):
            print("⚠️ Model not trained or loaded! Please train or load a model first.")
            return
        
        print("\n🔮 Disaster Prediction Interface")
        print("Enter the following environmental measurements:\n")
        
        user_input = []
        for feature in self.feature_names:
            while True:
                try:
                    value = float(input(f"{feature}: "))
                    user_input.append(value)
                    break
                except ValueError:
                    print("❌ Invalid input! Please enter a numeric value.")
        
        # Prepare input
        user_input = np.array(user_input).reshape(1, -1)
        user_input_scaled = self.scaler.transform(user_input)
        
        # Get original probabilities
        original_probs = self.model.predict_proba(user_input_scaled)[0]
        predicted_index = np.argmax(original_probs)
        predicted_label = self.label_encoder.inverse_transform([predicted_index])[0]
        predicted_probability = original_probs[predicted_index] * 100
        
        # Display results
        print("\n📊 Prediction Results:")
        print(f"🏷️ Most Likely Disaster: {predicted_label} ({predicted_probability:.1f}% confidence)")
        
        # Show all probabilities
        print("\n📈 Probability Distribution:")
        results = []
        for idx, prob in enumerate(original_probs):
            results.append([
                self.label_encoder.classes_[idx],
                f"{prob * 100:.1f}%",
                "✅" if idx == predicted_index else ""
            ])
        
        print(tabulate(results, 
                      headers=["Disaster Type", "Probability", "Predicted"], 
                      tablefmt="pretty"))
        
        # Visualize probabilities
        self.plot_probabilities(original_probs)

    def plot_probabilities(self, probabilities):
        """Visualize prediction probabilities."""
        plt.figure(figsize=(12, 6))
        colors = ['#ff9999' if i != np.argmax(probabilities) else '#66b3ff' for i in range(len(probabilities))]
        
        bars = plt.bar(self.label_encoder.classes_, probabilities, color=colors)
        plt.title('Disaster Type Probabilities', pad=20)
        plt.xlabel('Disaster Type')
        plt.ylabel('Probability')
        plt.xticks(rotation=45)
        plt.ylim(0, 1)
        
        # Add value labels
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.1%}',
                   ha='center', va='bottom')
        
        plt.tight_layout()
        plt.show()

    def run(self):
        """Main execution flow."""
        print("\n" + "="*60)
        print("🌪️ DISASTER PREDICTION SYSTEM".center(60))
        print("="*60 + "\n")
        
        # Load data
        file_path = "natural_disaster_dataset.xlsx"
        data = self.load_data(file_path)
        if data is None:
            return
        
        # Preprocessing
        X, y, self.feature_names = self.preprocess_data(data)
        y_encoded, self.label_encoder = self.encode_target(y)
        
        # Data splitting and balancing
        X_resampled, y_resampled = self.balance_data(X, y_encoded)
        X_train, X_test, y_train, y_test = self.split_data(X_resampled, y_resampled)
        X_train_scaled, X_test_scaled, self.scaler = self.normalize_features(X_train, X_test)
        
        # Store test data for evaluation
        self.X_test_scaled = X_test_scaled
        self.y_test = y_test
        
        # Model training
        self.train_model(X_train_scaled, y_train)
        self.train_xgboost(X_train_scaled, y_train)
        
        # Model comparison and selection
        self.compare_models()
        
        # Save the best model
        self.save_tools()
        
        # Interactive prediction
        while True:
            print("\n" + "-"*60)
            print("MENU:")
            print("1. Make a prediction")
            print("2. Exit")
            choice = input("Enter your choice (1-2): ")
            
            if choice == '1':
                self.predict_disaster()
            elif choice == '2':
                print("\n👋 Thank you for using the Disaster Prediction System!")
                break
            else:
                print("❌ Invalid choice! Please try again.")

if __name__ == "__main__":
    predictor = DisasterPredictor()
    predictor.run()